<a href="https://colab.research.google.com/github/asirabm/CSS-JS-scroll-animation/blob/main/feature_dataset_verification_and_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Feature Dataset Verification and Preparation**

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

prjt_path = "/content/drive/MyDrive/UCF_Crime_Project"
features_path = os.path.join(prjt_path, "features")

for split in ["train", "test"]:

    split_path = os.path.join(features_path, split)

    print(f"\n{split.upper()} DATASET")
    print("-" * 40)

    total = 0

    for category in sorted(os.listdir(split_path)):

        category_path = os.path.join(split_path, category)

        if os.path.isdir(category_path):

            count = len([
                f for f in os.listdir(category_path)
                if f.endswith(".npy")
            ])

            total += count

            print(f"{category:15}: {count}")

    print("-" * 40)
    print("Total feature files:", total)


TRAIN DATASET
----------------------------------------
Abuse          : 48
Arrest         : 45
Arson          : 41
Assault        : 47
Burglary       : 87
Explosion      : 29
Fighting       : 45
Normal         : 800
RoadAccidents  : 127
Robbery        : 145
Shooting       : 27
Shoplifting    : 29
Stealing       : 95
Vandalism      : 45
----------------------------------------
Total feature files: 1610

TEST DATASET
----------------------------------------
Abuse          : 2
Arrest         : 5
Arson          : 9
Assault        : 3
Burglary       : 13
Explosion      : 21
Fighting       : 5
Normal         : 150
RoadAccidents  : 23
Robbery        : 5
Shooting       : 23
Shoplifting    : 21
Stealing       : 5
Vandalism      : 5
----------------------------------------
Total feature files: 290


** Feature Dataset Preparation**

**Create the feature dataset metadata**

In [3]:
import os
import pandas as pd

project_path = "/content/drive/MyDrive/UCF_Crime_Project"
feature_path = os.path.join(project_path, "features")

records = []

for split in ["train", "test"]:

    split_path = os.path.join(feature_path, split)

    for category in sorted(os.listdir(split_path)):

        category_path = os.path.join(split_path, category)

        if not os.path.isdir(category_path):
            continue

        for filename in sorted(os.listdir(category_path)):

            if filename.endswith(".npy"):

                label = 0 if category == "Normal" else 1

                records.append({
                    "filename": filename,
                    "split": split,
                    "category": category,
                    "label": label,
                    "feature_path": os.path.join(category_path, filename)
                })

metadata_df = pd.DataFrame(records)

print("Total records:", len(metadata_df))
print("\nDataset distribution:")
print(metadata_df.groupby(["split", "label"]).size())

metadata_df.head()

Total records: 1900

Dataset distribution:
split  label
test   0        150
       1        140
train  0        800
       1        810
dtype: int64


,filename,split,category,label,feature_path
0,Abuse001_x264.npy,train,Abuse,1,/content/drive/MyDrive/UCF_Crime_Project/featu...
1,Abuse002_x264.npy,train,Abuse,1,/content/drive/MyDrive/UCF_Crime_Project/featu...
2,Abuse003_x264.npy,train,Abuse,1,/content/drive/MyDrive/UCF_Crime_Project/featu...
3,Abuse004_x264.npy,train,Abuse,1,/content/drive/MyDrive/UCF_Crime_Project/featu...
4,Abuse005_x264.npy,train,Abuse,1,/content/drive/MyDrive/UCF_Crime_Project/featu...


**label distribution**

In [4]:
print("\nLabel names:")
print(metadata_df["label"].value_counts().rename(index={
    0: "Normal",
    1: "Anomaly"
}))

print("\nMissing values:")
print(metadata_df.isnull().sum())


Label names:
label
Anomaly    950
Normal     950
Name: count, dtype: int64

Missing values:
filename        0
split           0
category        0
label           0
feature_path    0
dtype: int64


**Save Feature Dataset Metadata as csv in Drive**

In [5]:
metadata_path = os.path.join(
    project_path,
    "feature_metadata.csv"
)

metadata_df.to_csv(metadata_path, index=False)

print("Metadata saved successfully.")
print("File location:", metadata_path)

Metadata saved successfully.
File location: /content/drive/MyDrive/UCF_Crime_Project/feature_metadata.csv


In [6]:
print("File exists:", os.path.exists(metadata_path))

File exists: True


**Split Training Data into Training and Validation Sets**

**Create the split**

In [8]:
from sklearn.model_selection import train_test_split


train_metadata = metadata_df[metadata_df["split"] == "train"].copy()
test_metadata = metadata_df[metadata_df["split"] == "test"].copy()

# Spliting training data into training and validation sets
train_df, val_df = train_test_split(
    train_metadata,
    test_size=0.20,
    random_state=42,
    stratify=train_metadata["label"]
)

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Testing records:", len(test_metadata))

Training records: 1288
Validation records: 322
Testing records: 290


**Check class distribution**

In [9]:
for name, data in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_metadata)
]:
    print(f"\n{name} Dataset")
    print(data["label"].value_counts().sort_index().rename({
        0: "Normal",
        1: "Anomaly"
    }))


Training Dataset
label
Normal     640
Anomaly    648
Name: count, dtype: int64

Validation Dataset
label
Normal     160
Anomaly    162
Name: count, dtype: int64

Testing Dataset
label
Normal     150
Anomaly    140
Name: count, dtype: int64


**Save the three CSV files**

In [10]:
# output directory
split_path = os.path.join(project_path, "dataset_splits")
os.makedirs(split_path, exist_ok=True)

# Save each dataset split as train,validation and test
train_df.to_csv(
    os.path.join(split_path, "train_metadata.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(split_path, "validation_metadata.csv"),
    index=False
)

test_metadata.to_csv(
    os.path.join(split_path, "test_metadata.csv"),
    index=False
)

print("Dataset splits saved successfully.")

Dataset splits saved successfully.


**Verify the saved files**

In [13]:
print("Saved files in",split_path,":")

for filename in os.listdir(split_path):
    print(filename)

Saved files in /content/drive/MyDrive/UCF_Crime_Project/dataset_splits :
train_metadata.csv
validation_metadata.csv
test_metadata.csv
